# MoodMap Text Emotion Analyzer — Training Notebook

Fine-tunes RoBERTa-base on a **multi-dataset blend** (GoEmotions + dair-ai/emotion + TweetEval) to predict 9 emotion categories:

| Index | Emotion | Valence |
|-------|---------|--------|
| 0 | joy | positive |
| 1 | love | positive |
| 2 | optimism | positive |
| 3 | sadness | negative |
| 4 | anger | negative |
| 5 | fear | negative |
| 6 | disgust | negative |
| 7 | surprise | ambiguous |
| 8 | neutral | neutral |

3 positive vs 4 negative categories gives a balanced valence signal.

In [ ]:
!pip install -q transformers datasets evaluate scikit-learn accelerate

## 1. Load and Merge Datasets

We combine three complementary datasets:
- **GoEmotions** (58k Reddit comments, 28 labels → all 9 of our emotions — the backbone)
- **dair-ai/emotion** (20k Twitter messages → 6/9 emotions: joy, love, sadness, anger, fear, surprise)
- **TweetEval/emotion** (~5k tweets → 4/9 emotions: anger, joy, optimism, sadness)

### Why partial-label datasets don't simply zero-fill unmapped emotions

When a dair-ai sample is labelled "joy", the 3 emotions it never annotates
(optimism, disgust, neutral) are simply *unknown* — not confirmed absent.
Setting them to 0 would inject **false negatives** and teach the model incorrect
associations. Instead, unmapped positions are set to **-1 (masked)**, and the
loss function ignores those positions entirely. Only GoEmotions, which is a
fully multi-label annotated dataset, uses 0/1 for all 9 positions.

In [ ]:
from datasets import load_dataset, Dataset, DatasetDict, concatenate_datasets
from transformers import RobertaTokenizer
import numpy as np
from collections import Counter

TARGET_EMOTIONS = ["joy", "love", "optimism", "sadness", "anger", "fear", "disgust", "surprise", "neutral"]
NUM_LABELS = len(TARGET_EMOTIONS)

tokenizer = RobertaTokenizer.from_pretrained("roberta-base")

# ──────────────────────────────────────────────
# 1A. GoEmotions — label mapping to 9 categories
# ──────────────────────────────────────────────
# GoEmotions simplified labels (alphabetical):
#  0: admiration    7: curiosity    14: fear          21: pride
#  1: amusement     8: desire       15: gratitude     22: realization
#  2: anger         9: disappointment 16: grief       23: relief
#  3: annoyance    10: disapproval  17: joy           24: remorse
#  4: approval     11: disgust      18: love          25: sadness
#  5: caring       12: embarrassment 19: nervousness  26: surprise
#  6: confusion    13: excitement   20: optimism      27: neutral

GO_EMOTIONS_MAP = {
    # Joy (0): happiness, amusement, fun
    0: 0,   # admiration
    1: 0,   # amusement
    17: 0,  # joy
    23: 0,  # relief

    # Love (1): warmth, caring, gratitude, connection
    5: 1,   # caring
    15: 1,  # gratitude
    18: 1,  # love
    4: 1,   # approval

    # Optimism (2): excitement, hope, forward-looking positive
    13: 2,  # excitement
    20: 2,  # optimism
    21: 2,  # pride

    # Sadness (3): low-energy negative affect
    9: 3,   # disappointment
    12: 3,  # embarrassment
    16: 3,  # grief
    24: 3,  # remorse
    25: 3,  # sadness

    # Anger (4): high-energy hostile affect
    2: 4,   # anger
    3: 4,   # annoyance
    10: 4,  # disapproval

    # Fear (5): threat-response affect
    14: 5,  # fear
    19: 5,  # nervousness

    # Disgust (6): revulsion/aversion
    11: 6,  # disgust

    # Surprise (7): unexpected events
    26: 7,  # surprise
    22: 7,  # realization

    # Neutral (8)
    27: 8,  # neutral

    # Dropped (don't cleanly fit one category):
    #  6: confusion — cognitive state, not emotion
    #  7: curiosity — cognitive state, not emotion
    #  8: desire — motivational, maps poorly
}

def process_goemotions(batch):
    labels = np.zeros((len(batch["text"]), NUM_LABELS), dtype=np.float32)
    for i, raw_labels in enumerate(batch["labels"]):
        for lbl in raw_labels:
            if lbl in GO_EMOTIONS_MAP:
                labels[i][GO_EMOTIONS_MAP[lbl]] = 1.0

    tokenized = tokenizer(batch["text"], padding="max_length", truncation=True, max_length=128)
    tokenized["labels"] = labels.tolist()
    return tokenized

print("Loading GoEmotions...")
go_emotions = load_dataset("go_emotions", "simplified")
go_tok = go_emotions.map(process_goemotions, batched=True, remove_columns=go_emotions["train"].column_names)
print(f"  GoEmotions: {len(go_tok['train'])} train, {len(go_tok['validation'])} val, {len(go_tok['test'])} test")

In [ ]:
# ──────────────────────────────────────────────
# 1B. dair-ai/emotion — 6 single-label emotions
# ──────────────────────────────────────────────
# Labels: 0=sadness, 1=joy, 2=love, 3=anger, 4=fear, 5=surprise
#
# Known positions (annotated by this dataset):
#   joy(0), love(1), sadness(3), anger(4), fear(5), surprise(7)
# Masked positions (never annotated, set to -1):
#   optimism(2), disgust(6), neutral(8)
#
# Using -1 for unannotated positions prevents false-negative training
# signal. The masked BCE loss (Cell 10) will skip these positions.

DAIR_MAP = {
    0: 3,  # sadness  → sadness
    1: 0,  # joy      → joy
    2: 1,  # love     → love
    3: 4,  # anger    → anger
    4: 5,  # fear     → fear
    5: 7,  # surprise → surprise
}
DAIR_KNOWN = {0, 1, 3, 4, 5, 7}  # indices in TARGET_EMOTIONS that dair-ai annotates

def process_dair(batch):
    # Start everything at -1 (masked / unknown)
    labels = np.full((len(batch["text"]), NUM_LABELS), -1.0, dtype=np.float32)
    # Set known positions to 0 (not-present by default for this sample)
    for pos in DAIR_KNOWN:
        labels[:, pos] = 0.0
    # Set the one annotated emotion to 1 for each sample
    for i, lbl in enumerate(batch["label"]):
        if lbl in DAIR_MAP:
            labels[i][DAIR_MAP[lbl]] = 1.0

    tokenized = tokenizer(batch["text"], padding="max_length", truncation=True, max_length=128)
    tokenized["labels"] = labels.tolist()
    return tokenized

print("Loading dair-ai/emotion...")
dair = load_dataset("dair-ai/emotion", "split")
dair_tok = dair.map(process_dair, batched=True, remove_columns=dair["train"].column_names)
print(f"  dair-ai: {len(dair_tok['train'])} train, {len(dair_tok['validation'])} val, {len(dair_tok['test'])} test")

In [ ]:
# ──────────────────────────────────────────────
# 1C. TweetEval/emotion — 4 single-label tweet emotions
# ──────────────────────────────────────────────
# Labels: 0=anger, 1=joy, 2=optimism, 3=sadness
# Source: cardiffnlp/tweet_eval, config="emotion"
# Size: ~3,257 train / 374 val / 1,421 test
#
# Key advantage over SemEval-2018: proper Parquet format (no broken loading script),
# and provides direct "optimism" labels — the one category with least GoEmotions coverage.
#
# Known positions (annotated by this dataset):
#   joy(0), optimism(2), sadness(3), anger(4)
# Masked positions (never annotated, set to -1):
#   love(1), fear(5), disgust(6), surprise(7), neutral(8)

TWEETEVAL_MAP = {
    0: 4,  # anger    → anger
    1: 0,  # joy      → joy
    2: 2,  # optimism → optimism
    3: 3,  # sadness  → sadness
}
TWEETEVAL_KNOWN = {0, 2, 3, 4}  # indices in TARGET_EMOTIONS that TweetEval annotates

def process_tweeteval(batch):
    # Start everything at -1 (masked / unknown)
    labels = np.full((len(batch["text"]), NUM_LABELS), -1.0, dtype=np.float32)
    # Set known positions to 0 (not-present by default for this sample)
    for pos in TWEETEVAL_KNOWN:
        labels[:, pos] = 0.0
    # Set the one annotated emotion to 1 for each sample
    for i, lbl in enumerate(batch["label"]):
        if lbl in TWEETEVAL_MAP:
            labels[i][TWEETEVAL_MAP[lbl]] = 1.0

    tokenized = tokenizer(batch["text"], padding="max_length", truncation=True, max_length=128)
    tokenized["labels"] = labels.tolist()
    return tokenized

print("Loading TweetEval/emotion...")
tweeteval = load_dataset("cardiffnlp/tweet_eval", "emotion")
tweeteval_tok = tweeteval.map(process_tweeteval, batched=True, remove_columns=tweeteval["train"].column_names)
print(f"  TweetEval: {len(tweeteval_tok['train'])} train, {len(tweeteval_tok['validation'])} val, {len(tweeteval_tok['test'])} test")

In [ ]:
# ──────────────────────────────────────────────
# 1D. Merge all datasets
# ──────────────────────────────────────────────
# GoEmotions ships with a ClassLabel schema on its original 'labels'
# column. Even after map() overwrites it with float arrays, the Arrow
# schema may still carry the ClassLabel type. Explicit cast to
# Sequence(Value("float32")) on all three datasets before concatenation
# ensures schemas align so concatenate_datasets doesn't raise a
# "features can't be aligned" ValueError.

from datasets import Sequence, Value

LABEL_FEATURE = Sequence(Value("float32"))

for ds_dict in [go_tok, dair_tok, tweeteval_tok]:
    for split in ds_dict:
        ds_dict[split] = ds_dict[split].cast_column("labels", LABEL_FEATURE)

combined = DatasetDict({
    "train":      concatenate_datasets([go_tok["train"],      dair_tok["train"],      tweeteval_tok["train"]]),
    "validation": concatenate_datasets([go_tok["validation"], dair_tok["validation"], tweeteval_tok["validation"]]),
    "test":       concatenate_datasets([go_tok["test"],       dair_tok["test"],       tweeteval_tok["test"]]),
})

combined = combined.shuffle(seed=42)

print(f"\nCombined dataset:")
print(f"  GoEmotions : {len(go_tok['train']):>6,} train")
print(f"  dair-ai    : {len(dair_tok['train']):>6,} train")
print(f"  TweetEval  : {len(tweeteval_tok['train']):>6,} train")
print(f"  ─────────────────────────")
print(f"  Train total: {len(combined['train']):>6,}")
print(f"  Val total  : {len(combined['validation']):>6,}")
print(f"  Test total : {len(combined['test']):>6,}")
print(f"\n  Labels feature type: {combined['train'].features['labels']}")

## 2. Label Distribution Analysis

Check that no emotion class is severely underrepresented.

In [ ]:
import numpy as np

# Class weights MUST be computed from GoEmotions training data only.
# dair-ai and TweetEval use -1 for unannotated positions, so averaging
# over the full combined dataset would produce meaningless fractional
# counts for partially-covered emotions (e.g., optimism would appear
# far rarer than it actually is in the training signal).
#
# GoEmotions is fully annotated: every sample has a valid 0 or 1 for
# all 9 positions, so it gives a clean count of true positive/negative
# ratios per class.

go_train_labels = np.array(go_tok["train"]["labels"])  # shape: (N, 9), all 0/1
class_counts = go_train_labels.sum(axis=0)

print("GoEmotions training set label distribution (used for pos_weight):")
print("-" * 40)
for i, name in enumerate(TARGET_EMOTIONS):
    print(f"  {name:10s}: {int(class_counts[i]):>6,}")
print(f"  {'TOTAL':10s}: {int(class_counts.sum()):>6,}")

# pos_weight[i] = (# negative samples) / (# positive samples) for class i
# Tells BCE to penalise missing a rare positive class more than a common one.
total_go_samples = len(go_train_labels)
pos_weight = []
for i in range(NUM_LABELS):
    pos = class_counts[i]
    neg = total_go_samples - pos
    pos_weight.append(neg / max(pos, 1))

print(f"\nClass weights (pos_weight for BCEWithLogitsLoss, from GoEmotions only):")
for i, name in enumerate(TARGET_EMOTIONS):
    print(f"  {name:10s}: {pos_weight[i]:.2f}")

## 3. Model Setup with Weighted Loss

Key improvements:
- **Class-weighted BCEWithLogitsLoss** to handle label imbalance
- **Custom Trainer** that applies the weights during training
- **Learning rate warmup + cosine decay** for smoother convergence
- **Early stopping** to prevent overfitting

In [ ]:
import torch
import torch.nn as nn
from transformers import (
    RobertaForSequenceClassification,
    TrainingArguments,
    Trainer,
    DefaultDataCollator,
    EarlyStoppingCallback,
)

model = RobertaForSequenceClassification.from_pretrained(
    "roberta-base",
    num_labels=NUM_LABELS,
    problem_type="multi_label_classification",
)

class FloatLabelDataCollator(DefaultDataCollator):
    def __call__(self, features):
        batch = super().__call__(features)
        batch["labels"] = batch["labels"].to(torch.float32)
        return batch

class WeightedTrainer(Trainer):
    """
    Custom Trainer with two key improvements over standard BCE:

    1. Masked loss: positions where label == -1 are unknown (not annotated
       by the source dataset). We zero those positions out of the loss so
       the model receives no gradient signal for them — correct for partial-
       label datasets like dair-ai (3 masked) and TweetEval (5 masked).
       GoEmotions is fully annotated (all 0/1), so nothing is masked there.

    2. Class-weighted pos_weight: computed from GoEmotions counts only,
       so rare classes (e.g., disgust, neutral) are upweighted correctly.
    """
    def __init__(self, class_weights, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = torch.tensor(class_weights, dtype=torch.float32)

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")  # shape: (batch, 9), values in {-1, 0, 1}
        outputs = model(**inputs)
        logits = outputs.logits        # shape: (batch, 9)

        # Build a mask: True where we HAVE a valid label (0 or 1)
        mask = (labels != -1).float()  # shape: (batch, 9)

        # Replace -1 with 0 so BCE doesn't blow up on the fill value
        # (these positions will be zeroed out by the mask anyway)
        safe_labels = labels.clamp(min=0.0)

        # Element-wise BCE (reduction="none" gives per-position loss)
        loss_fn = nn.BCEWithLogitsLoss(
            pos_weight=self.class_weights.to(logits.device),
            reduction="none",
        )
        per_element_loss = loss_fn(logits, safe_labels)  # shape: (batch, 9)

        # Apply mask: zero out the unknown positions, then mean over known ones
        masked_loss = per_element_loss * mask
        # Avoid division by zero if an entire batch has no known labels (edge case)
        denom = mask.sum().clamp(min=1.0)
        loss = masked_loss.sum() / denom

        return (loss, outputs) if return_outputs else loss

In [ ]:
training_args = TrainingArguments(
    output_dir="./moodmap-roberta-results",

    # Learning rate with warmup
    learning_rate=2e-5,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",

    # Batch size and epochs
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    num_train_epochs=5,
    weight_decay=0.01,

    # Evaluation and saving
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    # Logging
    logging_steps=200,
    report_to="none",

    # Performance
    fp16=torch.cuda.is_available(),
    dataloader_num_workers=2,
)

trainer = WeightedTrainer(
    class_weights=pos_weight,
    model=model,
    args=training_args,
    train_dataset=combined["train"],
    eval_dataset=combined["validation"],
    processing_class=tokenizer,
    data_collator=FloatLabelDataCollator(),
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

print("Starting training...")
trainer.train()

## 4. Evaluation — Per-Class Metrics

Check F1, precision, and recall for each emotion individually.

In [ ]:
from sklearn.metrics import classification_report, multilabel_confusion_matrix
import numpy as np

# Evaluate on the GoEmotions test split only.
#
# Why not combined["test"]?
# The combined test set contains dair-ai and TweetEval samples that have
# -1 in unannotated positions. sklearn's classification_report would
# treat those -1 values as a third class, producing meaningless metrics.
# GoEmotions test is fully annotated (all 0/1 for all 9 emotions), so
# per-class F1/precision/recall are clean and interpretable there.

print("Evaluating on GoEmotions test set (fully annotated, all 9 labels)...")
predictions = trainer.predict(go_tok["test"])
probs = torch.sigmoid(torch.tensor(predictions.predictions)).numpy()

pred_labels = (probs >= 0.5).astype(int)
true_labels = np.array(predictions.label_ids)

# Sanity: confirm no -1 values in the GoEmotions test labels
assert (true_labels >= 0).all(), "Unexpected -1 in GoEmotions test labels!"

print("\nPer-Class Classification Report:")
print("=" * 60)
print(classification_report(
    true_labels,
    pred_labels,
    target_names=TARGET_EMOTIONS,
    digits=3,
    zero_division=0,
))

In [ ]:
mcm = multilabel_confusion_matrix(true_labels, pred_labels)

print("Per-Emotion Confusion Matrices (GoEmotions test set):")
print("=" * 50)
for i, name in enumerate(TARGET_EMOTIONS):
    tn, fp, fn, tp = mcm[i].ravel()
    total = tn + fp + fn + tp
    accuracy = (tn + tp) / total if total > 0 else 0.0
    print(f"\n{name}:")
    print(f"  True Neg: {tn:>5}  |  False Pos: {fp:>5}")
    print(f"  False Neg: {fn:>4}  |  True Pos:  {tp:>5}")
    print(f"  Class accuracy: {accuracy:.1%}")

## 5. Sanity Check — Test on Journal-Style Text

Verify the model handles mundane journal entries correctly before uploading.

In [ ]:
def predict_emotions(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=128)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model(**inputs)
    scores = torch.sigmoid(outputs.logits).squeeze().cpu().numpy()
    results = {TARGET_EMOTIONS[i]: round(float(scores[i]), 4) for i in range(NUM_LABELS)}
    dominant = max(results, key=results.get)

    # Compute valence the same way the backend will
    pos_avg = (results["joy"] + results["love"] + results["optimism"]) / 3.0
    neg_avg = (results["sadness"] + results["anger"] + results["fear"] + results["disgust"]) / 4.0
    neutral = results["neutral"]
    valence = (pos_avg - neg_avg) * (1.0 - 0.6 * neutral)
    valence = max(-1.0, min(1.0, valence))

    return results, dominant, round(valence, 4)

test_entries = [
    "Today was pretty average. I attended two lectures which were quite boring and I just wanted to leave.",
    "Had an amazing day! Got promoted at work and celebrated with friends over dinner.",
    "I feel so anxious about tomorrow's exam. I haven't studied enough and I'm scared I'll fail.",
    "Nothing much happened today. Just did some chores and watched TV.",
    "I'm furious. My roommate ate my food again without asking.",
    "Feeling really down today. Missing my family and feeling lonely.",
    "I can't believe what just happened - I won the lottery! This is unreal!",
    "Spent the evening with my partner, cooked dinner together. Feeling grateful for them.",
    "Things are looking up! Just got accepted into the programme I applied for. The future looks bright.",
]

print("Journal Entry Sanity Check")
print("=" * 80)
for entry in test_entries:
    scores, dominant, valence = predict_emotions(entry)
    print(f"\nEntry: \"{entry[:80]}...\"" if len(entry) > 80 else f"\nEntry: \"{entry}\"")
    print(f"  Dominant: {dominant} | Valence: {valence}")
    top3 = sorted(scores.items(), key=lambda x: -x[1])[:3]
    print(f"  Top 3: {', '.join(f'{k}={v:.3f}' for k, v in top3)}")
    if valence < -0.75:
        print(f"  ⚠️  WOULD TRIGGER DISTRESS AGENT")
    print()

## 6. Upload to Hugging Face

**Important:** Use an environment variable or Colab secret for your HF token — never hardcode it.

In [ ]:
from huggingface_hub import login
from google.colab import userdata

# Use Colab secrets (Settings → Secrets → Add "HF_TOKEN")
hf_token = userdata.get("HF_TOKEN")
login(token=hf_token, add_to_git_credential=True)

repo_name = "MNauman13/moodmap-roberta"

print(f"Uploading model to {repo_name}...")
model.push_to_hub(repo_name)
tokenizer.push_to_hub(repo_name)
print("Upload complete!")